# Model Experiment: ARIMA & SARIMA

**ამოცანა:** Walmart Store Sales Forecasting — Classical Statistical Time-Series მოდელები

ამ notebook-ში განვიხილავთ ARIMA და SARIMA მოდელებს, მათ თეორიულ საფუძვლებს და გამოვცდით
representative store-department წყვილებზე (და არა ყველა ~3000 სერიაზე, რადგან classical
statistical მოდელები ცალკეული სერიისთვის მაღალი გამოთვლითი ღირებულებისაა).

## რატომ ARIMA/SARIMA?

- **ARIMA (AutoRegressive Integrated Moving Average)** სამი კომპონენტისგან შედგება:
  - **AR(p)** — მიმდინარე მნიშვნელობა აიხსნება წინა *p* მნიშვნელობებით (რეგრესია საკუთარ წარსულზე)
  - **I(d)** — differencing-ის ხარისხი, რამდენჯერ უნდა დავადიფერენცოთ სერია რომ stationary გახდეს
  - **MA(q)** — მიმდინარე მნიშვნელობა აიხსნება წინა *q* ცდომილებით (residual errors)
- **SARIMA** = ARIMA + სეზონური კომპონენტი `(P, D, Q, s)`. ჩვენს შემთხვევაში `s=52`
  (კვირეული მონაცემი, წლიური სეზონურობა: Christmas, Thanksgiving, Super Bowl, Labor Day).

ARIMA არ ითვალისწინებს სეზონურობას საერთოდ, ამიტომ pure ARIMA-ს ველოდებით სუსტ შედეგს
holiday-heavy დატასეტზე — ეს არის მთავარი მოტივაცია SARIMA-ზე გადასვლისთვის.


In [ ]:
!pip install pmdarima wandb -q


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import wandb

from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.statespace.sarimax import SARIMAX

import pmdarima as pm


### `data_prep.py` და `evaluation.py`-ის ფუნქციები (inline)

რადგან Kaggle-ის environment-ს არ აქვს წვდომა repo-ს `src/` დირექტორიაზე,
`load_raw_data`, `merge_all`, `wmae`, `walk_forward_splits` ფუნქციები პირდაპირ
აქ არის ჩასმული (იდენტურია repo-ს `src/data_prep.py` და `src/evaluation.py`
ფაილების შემცველობის — თუ ორიგინალ ფაილებში რამე შეცვალე, ეს ასლიც განაახლე).

In [ ]:
def load_raw_data(data_dir="data"):
    train = pd.read_csv(f"{data_dir}/train.csv")
    test = pd.read_csv(f"{data_dir}/test.csv")
    features = pd.read_csv(f"{data_dir}/features.csv")
    stores = pd.read_csv(f"{data_dir}/stores.csv")

    train["Date"] = pd.to_datetime(train["Date"])
    test["Date"] = pd.to_datetime(test["Date"])
    features["Date"] = pd.to_datetime(features["Date"])

    return train, test, features, stores


def merge_all(sales_df, features_df, stores_df):
    df = sales_df.merge(features_df, on=["Store", "Date"], how="left", suffixes=("", "_feat"))
    df = df.merge(stores_df, on="Store", how="left")

    if "IsHoliday_feat" in df.columns:
        df = df.drop(columns=["IsHoliday_feat"])

    return df


def wmae(y_true, y_pred, is_holiday):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    is_holiday = np.asarray(is_holiday)

    weights = np.where(is_holiday, 5, 1)
    return np.sum(weights * np.abs(y_true - y_pred)) / np.sum(weights)


def walk_forward_splits(df, date_col="Date", n_splits=3, horizon_weeks=38):
    dates = pd.to_datetime(df[date_col])
    unique_dates = np.sort(dates.unique())

    horizon_days = horizon_weeks * 7
    splits = []
    last_date = unique_dates[-1]

    for i in range(n_splits):
        val_end = last_date - pd.Timedelta(days=horizon_days * i)
        val_start = val_end - pd.Timedelta(days=horizon_days)

        train_mask = dates < val_start
        val_mask = (dates >= val_start) & (dates <= val_end)

        if train_mask.sum() == 0:
            continue

        splits.append((train_mask.values, val_mask.values))

    return list(reversed(splits))


In [ ]:
from kaggle_secrets import UserSecretsClient
import os

os.environ["WANDB_API_KEY"] = UserSecretsClient().get_secret("WANDB_API_KEY")
wandb.login()

WANDB_PROJECT = "walmart-forecasting-statistical"


## 1. მონაცემების ჩატვირთვა და representative სერიების არჩევა

3000+ store-dept წყვილიდან ვირჩევთ რამდენიმეს, სხვადასხვა მახასიათებლით
(სხვადასხვა store type, სხვადასხვა საშუალო sales დონე), რომ თეორიული
დასკვნები საკმაოდ განზოგადებადი იყოს.

In [ ]:

KAGGLE_DATA_DIR = "/kaggle/input/competitions/walmart-recruiting-store-sales-forecasting"

def load_raw_data_kaggle(data_dir=KAGGLE_DATA_DIR):
    train = pd.read_csv(f"{data_dir}/train.csv.zip")
    test = pd.read_csv(f"{data_dir}/test.csv.zip")
    features = pd.read_csv(f"{data_dir}/features.csv.zip")
    stores = pd.read_csv(f"{data_dir}/stores.csv")

    train["Date"] = pd.to_datetime(train["Date"])
    test["Date"] = pd.to_datetime(test["Date"])
    features["Date"] = pd.to_datetime(features["Date"])

    return train, test, features, stores


train, test, features, stores = load_raw_data_kaggle()
df = merge_all(train, features, stores)
df = df.sort_values(["Store", "Dept", "Date"]).reset_index(drop=True)

# Representative store-dept წყვილები — შემცირებულია 5-დან 2-მდე, დროის დასაზოგად.
# მიზანია თეორიის რიცხვებით დასაბუთება, არა საუკეთესო შესაძლო score-ის ძებნა.
sample_pairs = [
    (1, 1),    # Store type A, popular dept — სტაბილური, მაღალი sales
    (20, 92),  # Store type A, high-volume dept — გამოხატული სეზონურობა
]

series_dict = {}
for store, dept in sample_pairs:
    s = df[(df.Store == store) & (df.Dept == dept)].set_index("Date")["Weekly_Sales"]
    s = s.asfreq("W-FRI")  # კვირეული სიხშირის ფიქსაცია, გაფ-ების NaN-ით შევსება
    series_dict[(store, dept)] = s

fig, axes = plt.subplots(len(sample_pairs), 1, figsize=(12, 3 * len(sample_pairs)), sharex=True)
for ax, (key, s) in zip(axes, series_dict.items()):
    ax.plot(s.index, s.values)
    ax.set_title(f"Store {key[0]}, Dept {key[1]}")
plt.tight_layout()
plt.show()


## 2. Stationarity ანალიზი (ADF ტესტი)

**Augmented Dickey-Fuller ტესტი** ამოწმებს null ჰიპოთეზას, რომ სერია არის non-stationary
(აქვს unit root). თუ p-value < 0.05, უარვყოფთ null ჰიპოთეზას — სერია stationary-ია.

ARIMA/SARIMA მოდელებს სჭირდებათ stationary სერია (ან differencing-ით მიღწეული stationarity),
რადგან AR/MA კომპონენტები predictable ავტოკორელაციურ სტრუქტურას ვარაუდობენ, რაც
non-stationary (trend/varying-variance) მონაცემში დარღვეულია.

In [ ]:
def adf_report(series, name=""):
    series_clean = series.dropna()
    result = adfuller(series_clean)
    print(f"--- {name} ---")
    print(f"ADF Statistic: {result[0]:.4f}")
    print(f"p-value: {result[1]:.4f}")
    print(f"Stationary (p<0.05): {result[1] < 0.05}")
    return result[1]

adf_pvalues = {}
for key, s in series_dict.items():
    adf_pvalues[key] = adf_report(s, name=f"Store {key[0]} / Dept {key[1]}")


In [ ]:
# პირველი დიფერენცირების ეფექტი stationarity-ზე (მაგალითი ერთ სერიაზე)
example_key = sample_pairs[0]
example_series = series_dict[example_key]

fig, axes = plt.subplots(2, 1, figsize=(12, 6))
axes[0].plot(example_series.index, example_series.values)
axes[0].set_title(f"Original series — Store {example_key[0]}, Dept {example_key[1]}")

diffed = example_series.diff().dropna()
axes[1].plot(diffed.index, diffed.values)
axes[1].set_title("After 1st differencing")
plt.tight_layout()
plt.show()

adf_report(diffed, name="After 1st differencing")


## 3. ACF / PACF გრაფიკები

ეს გრაფიკები გვეხმარება საწყისი `p` და `q` პარამეტრების შერჩევაში:
- **PACF** გვიჩვენებს AR(p) წევრის რიგს — მკვეთრი cutoff PACF-ში მიანიშნებს p-ზე
- **ACF** გვიჩვენებს MA(q) წევრის რიგს — მკვეთრი cutoff ACF-ში მიანიშნებს q-ზე

პრაქტიკაში, ავტომატური არჩევანისთვის (`auto_arima`) ამ გრაფიკებს უფრო
ინტერპრეტაციული/დიაგნოსტიკური დანიშნულებით ვიყენებთ.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
plot_acf(diffed, ax=axes[0], lags=40)
plot_pacf(diffed, ax=axes[1], lags=40)
plt.tight_layout()
plt.show()


## 4. ARIMA — auto_arima პარამეტრების შესარჩევად

`pmdarima.auto_arima` ავტომატურად ამოწმებს `(p,d,q)` კომბინაციების ბადეს და
AIC-ის მიხედვით ირჩევს საუკეთესოს. აქ სეზონურობას **არ** ვითვალისწინებთ
(`seasonal=False`) — ეს არის pure ARIMA baseline.

In [ ]:
def fit_auto_arima(series, seasonal=False, m=1):
    series_clean = series.dropna()
    model = pm.auto_arima(
        series_clean,
        start_p=0, start_q=0, max_p=3, max_q=3,
        d=None,               # ავტომატურად შეარჩევს KPSS/ADF ტესტით
        seasonal=seasonal,
        m=m,
        stepwise=True,
        suppress_warnings=True,
        error_action="ignore",
        trace=False,
    )
    return model

arima_models = {}
for key, s in series_dict.items():
    model = fit_auto_arima(s, seasonal=False)
    arima_models[key] = model
    print(f"Store {key[0]}, Dept {key[1]} -> order={model.order}, AIC={model.aic():.2f}")


## 5. ARIMA fit + walk-forward WMAE შეფასება + MLflow ლოგირება

თითოეული სერიისთვის ვიყენებთ არსებულ `walk_forward_splits`-ს (`src/evaluation.py`-დან),
ვთვლით WMAE-ს (holiday კვირებზე წონა 5) და ვლოგავთ MLflow-ში.

In [ ]:
def evaluate_arima_walk_forward(raw_df, store, dept, order, seasonal_order=(0, 0, 0, 0)):
    """
    ცალკეული store-dept წყვილისთვის walk-forward evaluation.
    raw_df — merge_all-ის შედეგი (გვჭირდება IsHoliday სვეტი წონისთვის).
    """
    sub = raw_df[(raw_df.Store == store) & (raw_df.Dept == dept)].sort_values("Date")
    splits = walk_forward_splits(sub, date_col="Date", n_splits=3, horizon_weeks=13)

    fold_scores = []
    for train_mask, val_mask in splits:
        train_part = sub[train_mask]
        val_part = sub[val_mask]
        if len(train_part) < 30 or len(val_part) == 0:
            continue

        y_train = train_part.set_index("Date")["Weekly_Sales"].asfreq("W-FRI")
        y_train = y_train.fillna(method="ffill")

        try:
            model = SARIMAX(
                y_train, order=order, seasonal_order=seasonal_order,
                enforce_stationarity=False, enforce_invertibility=False,
            ).fit(disp=False)
        except Exception as e:
            print(f"Fit failed for Store {store} Dept {dept}: {e}")
            continue

        n_periods = len(val_part)
        forecast = model.forecast(steps=n_periods)

        y_true = val_part["Weekly_Sales"].values
        y_pred = forecast.values[:len(y_true)]
        is_holiday = val_part["IsHoliday"].values

        score = wmae(y_true, y_pred, is_holiday)
        fold_scores.append(score)

    return np.mean(fold_scores) if fold_scores else None


arima_results = {}
for key, model in arima_models.items():
    store, dept = key
    order = model.order

    run = wandb.init(project=WANDB_PROJECT, name=f"ARIMA_Store{store}_Dept{dept}", reinit=True)
    wandb.config.update({"model_type": "ARIMA", "store": store, "dept": dept, "order": str(order)})

    score = evaluate_arima_walk_forward(df, store, dept, order=order)
    arima_results[key] = score

    wandb.log({"aic": model.aic(), "adf_pvalue": adf_pvalues[key]})
    if score is not None:
        wandb.log({"wmae": score})
    run.finish()

    print(f"Store {store}, Dept {dept} -> ARIMA order={order}, WMAE={score}")


## 6. SARIMA — სეზონურობის დამატება

**თეორია:** ARIMA-სგან განსხვავებით, SARIMA ცალკე მოდელირებს სეზონურ სტრუქტურას
`(P, D, Q, s)` პარამეტრებით. `s=52` ნიშნავს, რომ ვველით მსგავს შაბლონს ყოველ 52 კვირაში
(წელიწადში ერთხელ) — ეს პირდაპირ შეესაბამება Walmart-ის holiday-driven sales spikes-ს
(Christmas, Thanksgiving, Super Bowl, Labor Day ყოველწლიურად თითქმის იმავე კვირაში მეორდება).

პირველ რიგში ვხედავთ decomposition-ს (trend + seasonal + residual), რომ ვიზუალურად
დავრწმუნდეთ სეზონურობის არსებობაში.

**პრაქტიკული გადაწყვეტილება — fixed (P,D,Q,s), არა `auto_arima` ძებნა:**
`auto_arima(seasonal=True, m=52)`-ის stepwise search ძალიან ნელია (შესაძლოა ერთ სერიაზე
წუთები დასჭირდეს), რადგან თითოეული candidate model-ი სრულად fit-დება 52-სეზონურ lag-ზე.
რადგან დავალების მიხედვით მთავარი აქცენტი თეორიულ გარჩევაზეა და არა საუკეთესო score-ის
ძებნაზე, აქ ვირჩევთ გონივრულ, საწყის `(1,1,1)(0,1,1,52)` პარამეტრებს ხელით (`D=1` რადგან
სეზონური differencing ჩვეულებრივ საკმარისია ცხადი წლიური ციკლის მოსახსნელად,
`Q=1` სეზონურ ცდომილებათა კორელაციის დასაჭერად). ეს საკმარისია იმის საჩვენებლად,
რომ სეზონური კომპონენტი აუმჯობესებს pure ARIMA-ს შედეგს — რაც თეორიული დასკვნის
მთავარი მიზანია.

In [ ]:
example_series_filled = example_series.fillna(method="ffill").fillna(method="bfill")

decomposition = seasonal_decompose(example_series_filled, model="additive", period=52)
fig = decomposition.plot()
fig.set_size_inches(12, 8)
plt.tight_layout()
plt.show()


In [ ]:
def fit_fixed_sarima(series, order=(1, 1, 1), seasonal_order=(0, 1, 1, 52)):
    """
    ხელით დაფიქსირებული (P,D,Q,s) პარამეტრებით SARIMA fit — auto_arima-ს stepwise
    search-ის გვერდის ავლით (რაც m=52-ზე ძალიან ნელია). შედეგად ვღებულობთ
    სწრაფ, მაგრამ თეორიულად დასაბუთებულ SARIMA მოდელს შედარებისთვის.
    """
    series_clean = series.fillna(method="ffill").fillna(method="bfill")
    model = SARIMAX(
        series_clean,
        order=order,
        seasonal_order=seasonal_order,
        enforce_stationarity=False,
        enforce_invertibility=False,
    ).fit(disp=False)
    return model

sarima_models = {}
for key, s in series_dict.items():
    model = fit_fixed_sarima(s, order=(1, 1, 1), seasonal_order=(0, 1, 1, 52))
    sarima_models[key] = model
    print(f"Store {key[0]}, Dept {key[1]} -> order=(1,1,1), "
          f"seasonal_order=(0,1,1,52), AIC={model.aic:.2f}")


## 7. SARIMA fit + walk-forward WMAE + MLflow ლოგირება

In [ ]:
sarima_results = {}
for key in series_dict.keys():
    store, dept = key
    order = (1, 1, 1)
    seasonal_order = (0, 1, 1, 52)
    model = sarima_models[key]

    run = wandb.init(project=WANDB_PROJECT, name=f"SARIMA_Store{store}_Dept{dept}", reinit=True)
    wandb.config.update({
        "model_type": "SARIMA", "store": store, "dept": dept,
        "order": str(order), "seasonal_order": str(seasonal_order),
    })

    score = evaluate_arima_walk_forward(df, store, dept, order=order, seasonal_order=seasonal_order)
    sarima_results[key] = score

    wandb.log({"aic": model.aic})
    if score is not None:
        wandb.log({"wmae": score})
    run.finish()

    print(f"Store {store}, Dept {dept} -> SARIMA order={order}, "
          f"seasonal_order={seasonal_order}, WMAE={score}")


## 8. შედარება: ARIMA vs SARIMA

აქ ვხედავთ, რამდენად აუმჯობესებს სეზონური კომპონენტი WMAE-ს — ეს არის მთავარი
თეორიული დასკვნა.

In [ ]:
comparison = pd.DataFrame({
    "Store": [k[0] for k in sample_pairs],
    "Dept": [k[1] for k in sample_pairs],
    "ARIMA_WMAE": [arima_results.get(k) for k in sample_pairs],
    "SARIMA_WMAE": [sarima_results.get(k) for k in sample_pairs],
})
comparison["Improvement_%"] = (
    (comparison["ARIMA_WMAE"] - comparison["SARIMA_WMAE"]) / comparison["ARIMA_WMAE"] * 100
)
comparison
